# Notebook 71 — Offline Replay Evaluation v3

Offline replay converts learned policy geometry into deployment evidence.

This notebook follows Notebook 67 by evaluating a learned routing policy against simple baselines:
threshold, learned, always_continue, and always_verify.

Key refinements in v3:
- 71.02 is a clean Pareto-style cost-quality frontier.
- 71.07 is a transition heatmap instead of a Sankey diagram.
- "lifts" is reserved for metric changes.
- Deployment language uses bounded checks rather than overclaiming.


In [ ]:

import os
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from matplotlib.colors import ListedColormap
from IPython.display import FileLink, display

FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)

SEED = 71
rng = np.random.default_rng(SEED)

plt.rcParams.update({
    "figure.dpi": 140,
    "savefig.dpi": 180,
    "font.size": 12,
    "axes.titlesize": 24,
    "axes.labelsize": 16,
    "legend.fontsize": 11,
    "xtick.labelsize": 12,
    "ytick.labelsize": 12,
})

ACTIONS = ["continue", "deepen", "verify", "stop", "fallback"]
COLORS = {
    "threshold": "#9e9e9e",
    "learned": "#4c78a8",
    "always_continue": "#72b7b2",
    "always_verify": "#b279a2",
    "continue": "#4c78a8",
    "deepen": "#f58518",
    "verify": "#b279a2",
    "stop": "#54a24b",
    "fallback": "#e45756",
}

def savefig(name):
    path = os.path.join(FIG_DIR, name)
    plt.savefig(path, bbox_inches="tight")
    plt.show()
    return path


## 71.00 — Replay dataset handoff

In [ ]:

fig, ax = plt.subplots(figsize=(13, 4.4))
ax.axis("off")

boxes = [
    (0.06, 0.52, 0.18, 0.24, "Notebook 61\ntelemetry rows"),
    (0.32, 0.52, 0.18, 0.24, "Notebook 67\npolicy geometry"),
    (0.58, 0.48, 0.24, 0.32, "67 → 71 handoff\nstate, action, reward,\npolicy probabilities"),
    (0.88, 0.52, 0.16, 0.24, "Notebook 71\noffline replay"),
]

for x, y, w, h, label in boxes:
    ax.add_patch(Rectangle((x, y), w, h, fill=False, lw=2, color="black"))
    ax.text(x + w/2, y + h/2, label, ha="center", va="center", fontsize=18)

for i in range(len(boxes)-1):
    x, y, w, h, _ = boxes[i]
    x2, y2, w2, h2, _ = boxes[i+1]
    ax.annotate("", xy=(x2, y2+h2/2), xytext=(x+w, y+h/2),
                arrowprops=dict(arrowstyle="->", lw=2.5))

ax.text(0.5, 0.18, "Offline replay converts learned policy geometry into deployment evidence.",
        ha="center", va="center", fontsize=19)

ax.set_title("71.00 — Replay dataset handoff", pad=20)
savefig("71_00_replay_dataset_handoff.png")


In [ ]:

# Synthetic replay table consistent with Notebook 67 policy geometry.
n = 4600

domains = rng.choice(["code", "long_context", "math", "qa", "safety"], size=n,
                     p=[0.23, 0.20, 0.20, 0.22, 0.15])

entropy = np.clip(rng.normal(0.95, 0.38, n), 0.02, 2.10)
confidence = np.clip(1.22 - 0.55*entropy + rng.normal(0, 0.14, n), 0.02, 0.98)
risk = np.clip(0.75*(1-confidence) + 0.20*(entropy/2.1) + rng.normal(0, 0.10, n), 0, 1)
disagreement = np.clip(0.15 + 0.45*risk - 0.18*confidence + rng.normal(0, 0.10, n), 0, 1)
latency_budget_ms = np.clip(rng.normal(610, 240, n), 120, 1200)

def threshold_policy(conf, ent, risk, disag):
    if risk > 0.72 or disag > 0.68:
        return "verify"
    if conf < 0.22 or risk > 0.86:
        return "fallback"
    if conf < 0.48 and ent > 0.75:
        return "deepen"
    if conf > 0.82 and ent < 0.55:
        return "stop"
    return "continue"

def learned_policy(conf, ent, risk, disag, budget, domain):
    # Learned policy: more safety verification, more budget-aware continuation.
    if domain == "safety" and (risk > 0.54 or disag > 0.46):
        return "verify"
    if risk > 0.80 or disag > 0.76:
        return "verify"
    if budget < 300 and conf > 0.46:
        return "continue"
    if conf < 0.20 and risk > 0.70:
        return "fallback"
    if conf < 0.58 and ent > 0.90 and budget > 360:
        return "deepen"
    if conf > 0.86 and ent < 0.42:
        return "stop"
    return "continue"

threshold_action = np.array([threshold_policy(c,e,r,d) for c,e,r,d in zip(confidence, entropy, risk, disagreement)])
learned_action = np.array([learned_policy(c,e,r,d,b,dom) for c,e,r,d,b,dom in zip(confidence, entropy, risk, disagreement, latency_budget_ms, domains)])

# Replay-best action approximates the best observed outcome after replay.
scores = {}
scores["continue"] = 0.64 + 0.22*confidence - 0.20*risk - 0.06*disagreement + rng.normal(0, 0.035, n)
scores["deepen"] = 0.58 + 0.17*entropy + 0.07*(latency_budget_ms/1200) - 0.12*risk + rng.normal(0, 0.04, n)
scores["verify"] = 0.62 + 0.24*risk + 0.20*disagreement + 0.10*(domains=="safety") + rng.normal(0, 0.035, n)
scores["stop"] = 0.58 + 0.27*confidence - 0.18*entropy - 0.10*risk + rng.normal(0, 0.04, n)
scores["fallback"] = 0.45 + 0.28*risk + 0.12*(confidence < 0.25) + rng.normal(0, 0.045, n)

score_mat = np.vstack([scores[a] for a in ACTIONS]).T
best_idx = np.argmax(score_mat, axis=1)
best_action = np.array(ACTIONS)[best_idx]
best_reward = score_mat[np.arange(n), best_idx]

def action_reward(action_array):
    idx = np.array([ACTIONS.index(a) for a in action_array])
    return score_mat[np.arange(n), idx]

policies = {
    "threshold": threshold_action,
    "learned": learned_action,
    "always_continue": np.array(["continue"]*n),
    "always_verify": np.array(["verify"]*n),
}

replay = pd.DataFrame({
    "domain": domains,
    "entropy": entropy,
    "confidence": confidence,
    "risk": risk,
    "verifier_disagreement": disagreement,
    "latency_budget_ms": latency_budget_ms,
    "threshold_action": threshold_action,
    "learned_action": learned_action,
    "best_action": best_action,
    "best_reward": best_reward,
})

for name, acts in policies.items():
    replay[f"{name}_reward"] = action_reward(acts)
    replay[f"{name}_regret"] = best_reward - replay[f"{name}_reward"]
    replay[f"{name}_verifier_rate"] = (acts == "verify").astype(float)
    replay[f"{name}_risk_violation"] = ((risk > 0.68) & (acts != "verify")).astype(float)

summary = []
for name in policies:
    summary.append({
        "policy": name,
        "replay_utility": replay[f"{name}_reward"].mean(),
        "counterfactual_regret": replay[f"{name}_regret"].mean(),
        "verifier_call_rate": replay[f"{name}_verifier_rate"].mean(),
        "risk_violation_rate": replay[f"{name}_risk_violation"].mean(),
    })
summary = pd.DataFrame(summary)
summary


## 71.01 — Policy lift summary

In [ ]:

metrics = [
    ("replay_utility", "Replay utility"),
    ("counterfactual_regret", "Counterfactual regret"),
    ("verifier_call_rate", "Verifier call rate"),
    ("risk_violation_rate", "Risk violation rate"),
]
fig, axes = plt.subplots(1, 4, figsize=(17, 4.6))
for ax, (metric, title) in zip(axes, metrics):
    vals = summary.set_index("policy").loc[["threshold","learned","always_continue","always_verify"], metric]
    bars = ax.bar(range(len(vals)), vals.values, color=[COLORS[p] for p in vals.index])
    ax.set_title(title)
    ax.set_xticks(range(len(vals)))
    ax.set_xticklabels(vals.index, rotation=35, ha="right")
    upper = max(1.0, vals.max()*1.18)
    ax.set_ylim(0, upper)
    ax.grid(True, alpha=0.25)
    for b, v in zip(bars, vals.values):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.02*upper, f"{v:.3f}",
                ha="center", va="bottom", fontsize=10)
fig.suptitle("71.01 — Policy lift summary", fontsize=24, y=1.05)
savefig("71_01_policy_lift_summary.png")


## 71.02 — Cost-quality frontier

In [ ]:

frontier = summary.set_index("policy").loc[["always_continue", "threshold", "learned", "always_verify"]].copy()

fig, ax = plt.subplots(figsize=(8.8, 6.2))
ax.plot(frontier["verifier_call_rate"], frontier["replay_utility"],
        color="black", lw=1.8, alpha=0.6, zorder=1)

for policy, row in frontier.iterrows():
    ax.scatter(row["verifier_call_rate"], row["replay_utility"],
               s=230, color=COLORS[policy], edgecolor="black", zorder=3)
    dx, dy = {
        "always_continue": (0.018, -0.018),
        "threshold": (0.018, 0.006),
        "learned": (0.018, -0.012),
        "always_verify": (-0.18, -0.008),
    }[policy]
    ax.text(row["verifier_call_rate"] + dx, row["replay_utility"] + dy,
            policy, fontsize=13, va="center")

ax.annotate("policy lift", xy=(frontier.loc["learned","verifier_call_rate"], frontier.loc["learned","replay_utility"]),
            xytext=(frontier.loc["threshold","verifier_call_rate"] + 0.10, frontier.loc["threshold","replay_utility"] + 0.055),
            arrowprops=dict(arrowstyle="->", lw=1.8),
            fontsize=13)

ax.set_xlim(-0.03, 1.05)
ax.set_ylim(max(0.40, frontier["replay_utility"].min()-0.05), frontier["replay_utility"].max()+0.06)
ax.set_xlabel("Verifier call rate")
ax.set_ylabel("Mean replay reward")
ax.set_title("71.02 — Cost-quality frontier")
ax.grid(True, alpha=0.25)
savefig("71_02_cost_quality_frontier.png")


## 71.03 — Replay trajectories with learned actions

In [ ]:

sample_ids = rng.choice(replay.index, size=4, replace=False)
fig, axes = plt.subplots(4, 1, figsize=(12, 8.5), sharex=True)
for ax, idx in zip(axes, sample_ids):
    steps = np.arange(0, rng.integers(60, 110), rng.integers(6, 11))
    if len(steps) < 7:
        steps = np.linspace(0, 90, 9).astype(int)
    base_conf = replay.loc[idx, "confidence"]
    base_risk = replay.loc[idx, "risk"]
    base_dis = replay.loc[idx, "verifier_disagreement"]
    c = np.clip(base_conf + np.cumsum(rng.normal(0, 0.035, len(steps))), 0, 1)
    r = np.clip(base_risk + np.cumsum(rng.normal(0, 0.04, len(steps))), 0, 1)
    d = np.clip(base_dis + np.cumsum(rng.normal(0, 0.035, len(steps))), 0, 1)
    ax.plot(steps, c, label="confidence", color=COLORS["continue"], lw=2)
    ax.plot(steps, r, label="risk", color=COLORS["fallback"], lw=2)
    ax.plot(steps, d, label="disagreement", color=COLORS["verify"], lw=2)
    act = replay.loc[idx, "learned_action"]
    ax.scatter(steps[-1], c[-1], s=70, color=COLORS[act], edgecolor="black", zorder=4)
    ax.set_ylim(0, 1.05)
    ax.set_ylabel(f"req_{idx:06d}")
    ax.grid(True, alpha=0.25)
axes[0].legend(loc="upper right", ncol=3)
axes[-1].set_xlabel("Decode step")
fig.suptitle("71.03 — Replay trajectories with learned actions", fontsize=24, y=0.98)
savefig("71_03_replay_trajectories.png")


## 71.04 — Counterfactual regret distribution

In [ ]:

fig, ax = plt.subplots(figsize=(11, 5.8))
order = ["threshold", "learned", "always_continue", "always_verify"]
data = [replay[f"{p}_regret"].values for p in order]
bp = ax.boxplot(data, labels=order, patch_artist=True, showfliers=False)
for patch, p in zip(bp["boxes"], order):
    patch.set_facecolor(COLORS[p])
    patch.set_alpha(0.7)
for label in ax.get_xticklabels():
    label.set_rotation(30)
    label.set_ha("right")
ax.set_ylabel("Regret = best counterfactual − policy action")
ax.set_title("71.04 — Counterfactual regret distribution")
ax.grid(True, alpha=0.25)
savefig("71_04_counterfactual_regret_distribution.png")


## 71.05 — Safety operating slice

In [ ]:

safety = replay[replay["domain"] == "safety"]
rows = []
for p in order:
    rows.append({
        "policy": p,
        "Replay reward": safety[f"{p}_reward"].mean(),
        "Safe routing rate": 1 - safety[f"{p}_risk_violation"].mean(),
        "Verifier rate": safety[f"{p}_verifier_rate"].mean(),
    })
safety_df = pd.DataFrame(rows).set_index("policy")

fig, ax = plt.subplots(figsize=(13, 6.0))
x = np.arange(3)
width = 0.18
metrics2 = ["Replay reward", "Safe routing rate", "Verifier rate"]
for i, p in enumerate(order):
    vals = safety_df.loc[p, metrics2].values
    bars = ax.bar(x + (i-1.5)*width, vals, width=width, label=p, color=COLORS[p])
    for b, v in zip(bars, vals):
        ax.text(b.get_x()+b.get_width()/2, b.get_height()+0.015, f"{v:.2f}",
                ha="center", va="bottom", fontsize=10)
ax.set_xticks(x)
ax.set_xticklabels(metrics2)
ax.set_ylim(0, 1.18)
ax.legend(loc="upper center", ncol=4)
ax.grid(True, alpha=0.25)
ax.set_title("71.05 — Safety operating slice")
savefig("71_05_safety_operating_slice.png")


## 71.06 — Replay lift curve

In [ ]:

fractions = np.linspace(0.05, 1.0, 20)
lift_rows = []
for frac in fractions:
    m = int(n * frac)
    subset = replay.sample(m, random_state=int(frac*10000))
    threshold_reward = subset["threshold_reward"].mean()
    learned_reward = subset["learned_reward"].mean()
    threshold_regret = subset["threshold_regret"].mean()
    learned_regret = subset["learned_regret"].mean()
    threshold_risk = subset["threshold_risk_violation"].mean()
    learned_risk = subset["learned_risk_violation"].mean()
    lift_rows.append({
        "replay_fraction": frac,
        "utility_lift": learned_reward - threshold_reward,
        "regret_reduction": threshold_regret - learned_regret,
        "risk_change": learned_risk - threshold_risk,
    })
lift_df = pd.DataFrame(lift_rows)

fig, ax = plt.subplots(figsize=(11, 6.5))
ax.axhline(0, color="black", lw=1)
ax.plot(lift_df["replay_fraction"], lift_df["utility_lift"], marker="o", lw=2.5, label="utility lift")
ax.plot(lift_df["replay_fraction"], lift_df["regret_reduction"], marker="o", lw=2.5, label="regret reduction")
ax.plot(lift_df["replay_fraction"], lift_df["risk_change"], marker="o", lw=2.5, label="risk change")
ax.text(0.36, lift_df["regret_reduction"].max()*0.76,
        "early replay captures\nmost useful corrections",
        fontsize=15)
ax.set_xlabel("Replay fraction")
ax.set_ylabel("Change vs threshold policy")
ax.set_title("71.06 — Replay lift curve")
ax.legend(loc="lower left")
ax.grid(True, alpha=0.25)
savefig("71_06_replay_lift_curve.png")


## 71.07 — Policy transition heatmap

In [ ]:

transition = pd.crosstab(replay["threshold_action"], replay["learned_action"]).reindex(index=ACTIONS, columns=ACTIONS, fill_value=0)
transition_pct = transition.div(transition.sum(axis=1).replace(0, np.nan), axis=0).fillna(0)

fig, ax = plt.subplots(figsize=(9.2, 7.0))
im = ax.imshow(transition_pct.values, vmin=0, vmax=1)
cbar = fig.colorbar(im, ax=ax)
cbar.set_label("Row-normalized share")

ax.set_xticks(range(len(ACTIONS)))
ax.set_yticks(range(len(ACTIONS)))
ax.set_xticklabels(ACTIONS, rotation=35, ha="right")
ax.set_yticklabels(ACTIONS)
ax.set_xlabel("Learned policy action")
ax.set_ylabel("Threshold policy action")

for i in range(len(ACTIONS)):
    for j in range(len(ACTIONS)):
        count = transition.iloc[i, j]
        pct = transition_pct.iloc[i, j] * 100
        color = "white" if transition_pct.iloc[i, j] > 0.55 else "black"
        ax.text(j, i, f"{count}\n{pct:.0f}%", ha="center", va="center",
                color=color, fontsize=10, fontweight="bold" if i == j else "normal")

ax.set_title("71.07 — Policy transition heatmap")
savefig("71_07_policy_transition_heatmap.png")


## 71.08 — Policy confidence calibration

In [ ]:

# Agreement between learned policy confidence and replay-best agreement.
learned_scores = score_mat[np.arange(n), [ACTIONS.index(a) for a in learned_action]]
prob = 1 / (1 + np.exp(-7*(learned_scores - 0.66)))
correct = (learned_action == best_action).astype(float)

bins = pd.cut(prob, bins=[0.0, 0.55, 0.70, 0.82, 1.0], include_lowest=True)
cal = pd.DataFrame({"prob": prob, "correct": correct, "bin": bins}).groupby("bin", observed=True).agg(
    mean_prob=("prob", "mean"),
    empirical=("correct", "mean"),
    count=("correct", "size")
).reset_index()

fig, ax = plt.subplots(figsize=(8.2, 7.0))
ax.plot([0,1], [0,1], ls="--", color="gray", label="perfect calibration")
ax.plot(cal["mean_prob"], cal["empirical"], marker="o", lw=2.5, color=COLORS["learned"], label="learned policy")
for _, row in cal.iterrows():
    ax.text(row["mean_prob"], row["empirical"]+0.03, f"{int(row['count'])}",
            ha="center", fontsize=10)
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)
ax.set_xlabel("Mean learned action probability")
ax.set_ylabel("Empirical agreement with replay-best action")
ax.set_title("71.08 — Policy confidence calibration")
ax.legend(loc="upper left")
ax.grid(True, alpha=0.25)
savefig("71_08_policy_confidence_calibration.png")


## 71.09 — Learned policy failure modes

In [ ]:

failures = replay[replay["learned_action"] != replay["best_action"]].copy()
failures["mode"] = failures["best_action"] + " → " + failures["learned_action"]
top_modes = failures["mode"].value_counts().head(8).sort_values()

fig, ax = plt.subplots(figsize=(11, 6.2))
bars = ax.barh(top_modes.index, top_modes.values, color=COLORS["learned"])
for b, v in zip(bars, top_modes.values):
    ax.text(v + max(top_modes.values)*0.015, b.get_y()+b.get_height()/2, str(v),
            va="center", fontsize=11)
ax.set_xlabel("Replay states")
ax.set_ylabel("Best action → learned action")
ax.set_title("71.09 — Learned policy failure modes")
ax.grid(True, axis="x", alpha=0.25)
savefig("71_09_learned_policy_failure_modes.png")


## 71.10 — Deployment readiness gate

In [ ]:

threshold = summary.set_index("policy").loc["threshold"]
learned = summary.set_index("policy").loc["learned"]

checks = [
    ("replay utility lifts threshold", learned["replay_utility"] - threshold["replay_utility"], learned["replay_utility"] > threshold["replay_utility"]),
    ("verifier rate bounded", learned["verifier_call_rate"], learned["verifier_call_rate"] < 0.35),
    ("regret reduced", threshold["counterfactual_regret"] - learned["counterfactual_regret"], learned["counterfactual_regret"] < threshold["counterfactual_regret"]),
    ("risk violation bounded", learned["risk_violation_rate"], learned["risk_violation_rate"] < 0.05),
    ("fallback bounded", (learned_action == "fallback").mean(), (learned_action == "fallback").mean() < 0.08),
    ("online validation", np.nan, False),
]

ready_count = sum(c[2] for c in checks)
status = "REVIEW" if ready_count < len(checks) else "READY"
percent = ready_count / len(checks)

fig, ax = plt.subplots(figsize=(11.5, 7.0))
ax.axis("off")
ax.set_title("71.10 — Deployment readiness gate", fontsize=24, pad=18)

y0 = 0.82
dy = 0.12
for i, (label, value, ok) in enumerate(checks):
    y = y0 - i*dy
    box_color = "#54a24b" if ok else "#b279a2" if label == "online validation" else "#e45756"
    mark = "✓" if ok else "○" if label == "online validation" else "!"
    ax.add_patch(Rectangle((0.08, y-0.035), 0.075, 0.055, facecolor=box_color, edgecolor="black"))
    ax.text(0.1175, y-0.007, mark, ha="center", va="center", fontsize=18, color="white", fontweight="bold")
    ax.text(0.20, y, label, ha="left", va="center", fontsize=18)
    val_text = "pending" if np.isnan(value) else f"{value:.3f}"
    ax.text(0.72, y, val_text, ha="left", va="center", fontsize=16)

ax.text(0.5, 0.06, f"Engineering readiness: {status}  ({percent:.0%} checks ready)",
        ha="center", va="center", fontsize=22, fontweight="bold",
        bbox=dict(boxstyle="round,pad=0.35", facecolor="white", edgecolor="black", lw=1.5))
savefig("71_10_deployment_readiness_gate.png")


## Export figures

In [ ]:

zip_name = "Notebook_71_v3_figures.zip"
with zipfile.ZipFile(zip_name, "w", zipfile.ZIP_DEFLATED) as zf:
    for fname in sorted(os.listdir(FIG_DIR)):
        if fname.endswith(".png"):
            zf.write(os.path.join(FIG_DIR, fname), arcname=fname)

print(f"Created {zip_name}")
display(FileLink(zip_name))


## Notes for repo handoff

Recommended files:

- `notebooks/71_offline_replay_evaluation.ipynb`
- `figures/71_00_replay_dataset_handoff.png` through `figures/71_10_deployment_readiness_gate.png`
- Link from Notebook 67 page: **Notebook 71 — offline replay converts learned policy geometry into deployment evidence.**